In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import spectrum as sp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
date = "20261117"

In [ ]:
run_range = [54, 56, 58, 60, 61, 62]
colors = plt.get_cmap("viridis")(np.linspace(0, 0.8, len(run_range)))
for idx, acq_no in enumerate(run_range):
    acq = daq.load_acq(date, acq_no, calibrated=True).normalised().smooth_gauss(0.05)
    l = f"acq: {acq.meta['acq']}, y = {acq.meta['a_mp1_y']:.0f}mm"
    plt.plot(acq.energy, acq.intensity, label=l, color=colors[idx])
plt.legend()
plt.ylim(0, 0.5)
plt.xlim(526, 529)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

## Combination comparison

In [ ]:
import numpy as np

ACQS, LO, HI = (62, 63), 526, 529
runs = [daq.load_acq("20261117", a, calibrated=True).binned(8) for a in ACQS]
total = sp.combine(runs)

fig, (ax, axr) = plt.subplots(2, 1, sharex=True, height_ratios=[3, 1])
for run in runs:
    area = np.trapezoid(run.intensity, x=run.energy)
    y, dy = run.intensity / area, np.sqrt(np.abs(run.intensity).clip(1)) / area
    ax.fill_between(run.energy, y - dy, y + dy, alpha=0.4, label=f"acq: {run.meta['acq']}")
total.normalised().plot(ax, color="k", label="combined")
ax.set_ylim(0, 0.5)
ax.set_ylabel("Intensity (a.u.)")
ax.legend()

for chi2, pull in sp.compare(runs, lo=LO, hi=HI):
    pull.plot(axr, lw=0.8)
    print(f"acq {pull.meta['acq']}: chi2_red {chi2:.2f}")
axr.axhspan(-1, 1, color="0.85")
axr.set_ylabel(r"pull ($\sigma$)")
axr.set_xlabel("Energy (eV)")
axr.set_xlim(LO, HI)
plt.show()

for target in (0.10, 0.05, 0.03):
    print(f"runs needed for {target:.0%} error per point: {sp.runs_needed(runs, target, lo=LO, hi=HI):.1f}")